# Notebook 2: Feature Engineering
## CS336 - Artificial Intelligence and Machine Learning (AIML)
### Assignment: Smart Energy Consumption Advisory Agent

**Purpose:** This notebook transforms the cleaned smart meter data into a rich set of
features that capture temporal patterns, rolling statistics,...

In [ ]:
# ─── Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA          # For dimensionality reduction
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (13, 5)
print('Imports successful.')

## 1. Load Preprocessed Data

In [ ]:
# ─── Load the clean dataset produced by Notebook 1 ───────────────────────────
# parse_dates ensures the timestamp column is treated as datetime
df = pd.read_csv('data_clean.csv', parse_dates=['timestamp'], index_col='timestamp')

print(f'Loaded {df.shape[0]} rows, {df.shape[1]} columns.')
df.head(3)

## 2. Rolling Window Statistics

Rolling features capture **short-term trends** that a single snapshot cannot reveal.

In [ ]:
# ─── 1-hour rolling mean and standard deviation ──────────────────────────────
# min_periods=1 ensures we get values even at the start of the series
window = 4  # 4 × 15-minute intervals = 1 hour

df['rolling_mean_1h'] = df['global_active_power'].rolling(window, min_periods=1).mean()
df['rolling_std_1h']  = df['global_active_power'].rolling(window, min_periods=1).std().fillna(0)

# Also compute a 24-hour (96-interval) rolling mean for long-term trend
df['rolling_mean_24h'] = df['global_active_power'].rolling(96, min_periods=1).mean()

print('Rolling features added: rolling_mean_1h, rolling_std_1h, rolling_mean_24h')

## 3. Lag Features

Lag features expose the model to **historical context**.

In [ ]:
# ─── Lag features: consumption 1, 4, and 96 intervals ago ───────────────────
# lag_1  = 15 minutes ago   → captures inertia (appliances stay on)
# lag_4  = 1 hour ago       → captures hourly cycles
# lag_96 = 24 hours ago     → captures same-time-yesterday pattern
for lag in [1, 4, 96]:
    col_name = f'lag_{lag}'
    df[col_name] = df['global_active_power'].shift(lag)

# Drop the small number of rows that now have NaN due to shifting
df.dropna(inplace=True)
print(f'After lag feature construction: {df.shape[0]} rows remain.')

## 4. Time-of-Use Segments

Electricity tariffs and behaviour differ by **time-of-use (ToU)** period.

In [ ]:
# ─── Time-of-use segmentation ────────────────────────────────────────────────
def assign_tou(hour):
    """Return a ToU label based on the hour of the day.

    Off-peak  : 22:00 – 07:59   (low tariff, typically night)
    Shoulder  : 08:00 – 16:59   (moderate tariff, daytime)
    Peak      : 17:00 – 21:59   (high tariff, evening demand surge)
    """
    if hour < 8 or hour >= 22:
        return 'off_peak'
    elif hour < 17:
        return 'shoulder'
    else:
        return 'peak'

df['tou_segment'] = df.index.hour.map(assign_tou)  # Apply to each row

# One-hot encode the categorical ToU segment for use in ML models
tou_dummies = pd.get_dummies(df['tou_segment'], prefix='tou', drop_first=False)
df = pd.concat([df, tou_dummies], axis=1)

print('ToU segment distribution:')
print(df['tou_segment'].value_counts())

## 5. Daily Usage Profile

Aggregating to daily statistics gives a **household-level signature** — 
useful for clustering households by behaviour.

In [ ]:
# ─── Daily aggregate features ────────────────────────────────────────────────
# Group readings by calendar date and compute summary statistics
daily = df['global_active_power'].resample('D').agg(
    daily_total='sum',    # Total energy consumed that day (proportional to kWh)
    daily_mean='mean',    # Average power draw
    daily_max='max',      # Peak power draw
    daily_std='std'       # Variability (high std → erratic usage)
).fillna(0)

print(f'Daily aggregate table: {daily.shape}')
daily.head()

## 6. Feature Importance via Variance

We use **explained variance** from PCA as a proxy for how much information each feature contributes.

In [ ]:
# ─── PCA on numeric features to assess variance contribution ────────────────
numeric_cols = ['global_active_power', 'sub_metering_1', 'sub_metering_2',
                'sub_metering_3', 'rolling_mean_1h', 'rolling_std_1h',
                'lag_1', 'lag_4', 'lag_96', 'hour_sin', 'hour_cos']

X = df[numeric_cols].fillna(0).values

# Standardise before PCA — PCA is sensitive to scale differences
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit PCA retaining all components to study the explained variance spectrum
pca = PCA()
pca.fit(X_scaled)

# Plot cumulative explained variance to find the 'elbow'
cumvar = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(cumvar) + 1), cumvar, marker='o', color='teal')
plt.axhline(0.95, color='salmon', linestyle='--', label='95 % threshold')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Explained Variance Spectrum')
plt.legend()
plt.tight_layout()
plt.show()

# Report how many PCs are needed for 95 % variance
n95 = np.argmax(cumvar >= 0.95) + 1
print(f'{n95} components explain ≥ 95 % of variance.')

## 7. 2-D PCA Projection for Visual Inspection

In [ ]:
# ─── Project all data points onto first two PCs ──────────────────────────────
# Two PCs allow us to visualise high-dimensional patterns in 2-D
pca2 = PCA(n_components=2)
X2 = pca2.fit_transform(X_scaled)

# Colour by time-of-use segment to reveal if segments separate in PCA space
tou_map = {'off_peak': 0, 'shoulder': 1, 'peak': 2}
colours = df['tou_segment'].map(tou_map).values

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X2[:, 0], X2[:, 1],
                      c=colours, cmap='viridis', alpha=0.3, s=3)
plt.colorbar(scatter, label='ToU Segment (0=off-peak, 1=shoulder, 2=peak)')
plt.title('2-D PCA Projection Coloured by Time-of-Use Segment')
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Save engineered dataset for Notebooks 3 and 4 ───────────────────────────
df.to_csv('data_engineered.csv')    # Full feature matrix
daily.to_csv('data_daily.csv')      # Daily aggregate profiles

print('Saved: data_engineered.csv')
print('Saved: data_daily.csv')
print(f'Feature count: {df.shape[1]}')